# 07 — Attribution synthesis → shortlist

Borda on **selection** importances. Also: rank agreement + model comparison on selection. Final 2024 is untouched here.


In [ ]:
from spy_alpha_research.data import (
    load_config, project_root, load_frame, save_frame, split_three_way, make_target,
)
ROOT = project_root()
exp = load_config(ROOT / "configs" / "experiment.yaml")
models = load_config(ROOT / "configs" / "models.yaml")
bt = load_config(ROOT / "configs" / "backtest.yaml")
FIG = ROOT / exp["paths"]["figures_dir"]
FIG.mkdir(parents=True, exist_ok=True)
PROC = ROOT / exp["paths"]["processed_dir"]
dates = exp["dates"]

from spy_alpha_research.pipelines import fit_random_forest, fit_pca_ridge
from spy_alpha_research.attribution import (
    select_alpha_candidates, save_alpha_candidates, ranking_agreement_table,
)
from spy_alpha_research.reporting import (
    plot_importance, plot_rank_stability, compare_predictions, plot_comparison_table_bars,
)

X_train = load_frame(PROC / "features_train_scaled.parquet")
X_sel = load_frame(PROC / "features_selection_scaled.parquet")
y = load_frame(PROC / "spy_target.parquet").iloc[:, 0]
rf = fit_random_forest(
    X_train, y.reindex(X_train.index), X_sel, y.reindex(X_sel.index),
    random_state=exp["project"]["seed"], **models["trees"]["random_forest"],
)
feats = load_frame(PROC / "features_raw.parquet")
train_idx, sel_idx, final_idx = split_three_way(
    feats.index,
    train_end=dates["train_end"],
    selection_start=dates["selection_start"],
    selection_end=dates["selection_end"],
    final_start=dates["final_start"],
    end=dates["end"],
)
ridge = fit_pca_ridge(
    feats.loc[train_idx], y.reindex(train_idx),
    feats.loc[sel_idx], y.reindex(sel_idx),
    n_components=models["pca_kernel"]["n_components"],
    alpha=models["pca_kernel"]["ridge_alpha"],
    scale=True, min_periods=exp["scaling"]["min_periods"],
)
imp_map = {"rf": rf.importances, "pca_ridge": ridge.feature_attribution.abs()}
agree = ranking_agreement_table(imp_map)
plot_rank_stability(agree, FIG / "rank_stability.png")
names = select_alpha_candidates(
    imp_map, top_k=exp["alpha"]["top_k"], method=exp["alpha"]["consensus_method"]
)
save_alpha_candidates(
    names, ROOT / exp["paths"]["alpha_candidates"],
    meta={"method": exp["alpha"]["consensus_method"], "window": "selection_2022_2023"},
)

# Selection-window predictive comparison (not final)
train_mean = float(y.reindex(train_idx).mean())
pred_cmp = compare_predictions(
    {"rf": rf.eval_predictions, "pca_ridge": ridge.eval_predictions},
    y.reindex(sel_idx),
    train_mean=train_mean,
)
plot_comparison_table_bars(pred_cmp, "r2_vs_train_mean", FIG / "model_comparison.png")
names, agree, pred_cmp
